Vector Stores


What is vector indexing?
Suppose you have one million vectors. Comparing the query against every stored vector can be expensive.

A vector index organizes vectors to make nearest-neighbor searches more efficient.

There are two important indexing strategies in our syllabus: IVF and HNSW.

IVF — Inverted File Index
IVF groups vectors into clusters. At query time, it searches selected nearby clusters instead of scanning the entire collection.

Main trade-off: faster search with fewer clusters examined, but relevant vectors may be missed.

HNSW — Hierarchical Navigable Small World
HNSW organizes vectors into a graph with multiple navigation layers. Search traverses the graph toward promising candidate vectors.

Main trade-off: often offers fast, high-recall search, but uses additional memory and requires index construction.

Embedding model: creates vectors.



Vector store: stores vectors and retrieves records.



Vector index: accelerates nearest-neighbor search.



Similarity or distance metric: determines how closeness is measured.



Retrieval relevance: whether the retrieved content actually helps answer the user's question

Vector Store Operations — Create, Read, Update, Delete (CRUD).|

In [1]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

documents = [
    Document(
        page_content="Python is a popular programming language.",
        metadata={"source": "python_guide", "version": 1}
    ),
    Document(
        page_content="FAISS enables efficient similarity search over vectors.",
        metadata={"source": "faiss_guide", "version": 1}
    )
]

vector_store = FAISS.from_documents(documents, embeddings)

C:\Users\aksha\AppData\Local\Temp\ipykernel_21760\3568400721.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
results = vector_store.similarity_search(
    "How can i search vectors efficiently",
    k=2
)

In [3]:
results

[Document(id='f7a8866c-c464-41a0-899e-57b722f950a4', metadata={'source': 'faiss_guide', 'version': 1}, page_content='FAISS enables efficient similarity search over vectors.'),
 Document(id='22233fb8-6c66-4eea-9f0d-5c91a0741384', metadata={'source': 'python_guide', 'version': 1}, page_content='Python is a popular programming language.')]

In [4]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)

FAISS enables efficient similarity search over vectors.
{'source': 'faiss_guide', 'version': 1}
Python is a popular programming language.
{'source': 'python_guide', 'version': 1}


In [8]:
results_with_scores= vector_store.similarity_search_with_score(
"How can i search vectors efficiently",
k=2
)


In [12]:
for doc,dist in results_with_scores:

    print(doc)
    print(dist)

page_content='FAISS enables efficient similarity search over vectors.' metadata={'source': 'faiss_guide', 'version': 1}
0.7460675
page_content='Python is a popular programming language.' metadata={'source': 'python_guide', 'version': 1}
1.9210434


In [13]:
updated_document = Document(
    page_content="Python is a general-purpose programming language.",
    metadata={"source": "python_guide", "version": 2}
)

vector_store.add_documents(
    [updated_document],
    ids=["python-guide-v1"]
)

['python-guide-v1']

Important correction: add_documents() with an existing ID updates the associated record in the LangChain FAISS integration when that ID is present. If the ID doesn't exist, the operation adds a record. Therefore, choose stable IDs and handle ID collisions deliberately.

Also, the ID used here must exist in the store. Our initial example did not assign "python-guide-v1" as an ID, so to make this example fully executable, we should assign IDs when creating the original store:


vector_store = FAISS.from_documents(    documents,    embeddings,    ids=["python-guide-v1", "faiss-guide-v1"])



Then the update can target "python-guide-v1".

When document content changes, its embedding must also be regenerated. Updating only metadata does not update the semantic representation of changed text.

A complete CRUD example

In [19]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docs = [
    Document(
        page_content="Python is a popular programming language.",
        metadata={"source": "python_guide"}
    ),
    Document(
        page_content="FAISS enables efficient vector search.",
        metadata={"source": "faiss_guide"}
    )
]

store = FAISS.from_documents(
    docs,
    embeddings,
    ids=["python-guide", "faiss-guide"]
)

# READ: Search
print(store.similarity_search("vector search", k=2))

# UPDATE: Replace the existing document and regenerate its vector
store.delete(ids=["python-guide"])
store.add_documents(
    [
        Document(
            page_content="Python is a general-purpose programming language.",
            metadata={"source": "python_guide"}
        )
    ],
    ids=["python-guide"]
)

# DELETE: Remove a document
store.delete(ids=["faiss-guide"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Document(id='faiss-guide', metadata={'source': 'faiss_guide'}, page_content='FAISS enables efficient vector search.'), Document(id='python-guide', metadata={'source': 'python_guide'}, page_content='Python is a popular programming language.')]


True

What happens if deleted source documents remain in the vector store?
Partially correct
You mentioned duplication, but the primary problem is stale retrieval. Duplication is possible in some ingestion workflows, but it isn't the inevitable consequence of leaving an old vector behind.

Example: You delete an outdated refund policy from the source knowledge base, but its embedding remains indexed. A user asks about refunds, and the retriever returns the old policy.

Interview-ready answer:

“Leaving vectors for deleted source documents creates stale knowledge in the vector store. The retriever may return outdated or invalid information, causing incorrect answers. We need to synchronize deletions between the source data and the vector store.”